In [1]:
%load_ext autoreload
%autoreload 2

from paper_utils import *
    
import os
os.chdir('../..')
from sklearn.metrics import cohen_kappa_score


In [17]:
runs = {
    'uvfbxm6d': ['squad-llama2', ['config.yaml', 'validation_generations.pkl']],
    'm3y3x605': ['squad-gpt35', ['config.yaml', 'uncertainty_measures.pkl']],
    # gpt-4 is running at 170816
}

all_configs, all_results = {}, {}
for wandb_id, (name, files) in runs.items():
    all_configs[wandb_id], all_results[wandb_id] = restore_file(wandb_id, filenames=files)

In [66]:
active_run = 'uvfbxm6d'

N_ANNOTATIONS = 100
START_i = 0
END_i = 100
users = ['jansen', 'sebhar']

# Enable this to also get gpt/llama2 truth data in the CSV
WRITE_AUTOMATED_METRICS = False


In [121]:
columns = ['index', 'data_id', 'wandb_id', 'dataset', 'truth_metric', 'truth_value']

runs_data = []

for i in range(START_i, END_i):

    key = list(all_results[active_run].keys())[i]
    
    result = all_results[active_run][key]
    print(80 * f'-')
    print(f'{i} / {key} -------- New Question ------')
    print('Q:', result['question'])
    print('True answer:', result['reference']['answers']['text'])
    print('Model response:', result['most_likely_answer']['response'])
    
    for run in runs:
        if runs[run][1][1] == 'uncertainty_measures.pkl':
            accuracy = 1 - all_results[run]['validation_is_false'][i]
        elif runs[run][1][1] == 'validation_generations.pkl':
            accuracy = result['most_likely_answer']['accuracy']
        else:
            raise
        accuracy = int(accuracy)
        dataset, metric = runs[run][0].split('-')
        run_data = [i, key, run, dataset, metric, accuracy]
        print(f'Accuracy {metric}: {accuracy}')

        if WRITE_AUTOMATED_METRICS:
            runs_data.append(run_data)

        print(','.join([str(i) for i in run_data]))

    print(f'Accuracy human: ???')
    for user in users:
        runs_data.append([i, key, run, dataset, f'human_{user}', 'FILL_IN'])

--------------------------------------------------------------------------------
0 / 57338007d058e614000b5bdb -------- New Question ------
Q: What was Warsaw's population in 1901?
True answer: ['711,988', '711,988', '711,988']
Model response: According to the 1901 census, Warsaw's population was approximately 750,000 people.
Accuracy llama2: 0
0,57338007d058e614000b5bdb,uvfbxm6d,squad,llama2,0
Accuracy gpt35: 0
0,57338007d058e614000b5bdb,m3y3x605,squad,gpt35,0
Accuracy human: ???
--------------------------------------------------------------------------------
1 / 571cc5c45efbb31900334dde -------- New Question ------
Q: When did O2 begin to acculturate in the atmosphere?
True answer: ['2.5 billion years ago', '2.5 billion years ago', 'about 2.5 billion years ago', 'about 2.5 billion years ago', '2.5 billion years ago during the Great Oxygenation Event']
Model response: O2 began to accumulate in the atmosphere around 2.7 billion years ago during the Great Oxygenation Event.
Accuracy llam

# Use this to set up the CSV that you fill values in

In [101]:
print(pd.DataFrame(runs_data, columns=columns).sort_values(['truth_metric', 'index']).to_csv(index=False))

index,data_id,wandb_id,dataset,truth_metric,truth_value
0,57338007d058e614000b5bdb,m3y3x605,squad,gpt35,0
1,571cc5c45efbb31900334dde,m3y3x605,squad,gpt35,0
2,5733a5f54776f41900660f46,m3y3x605,squad,gpt35,1
3,5727dd2e4b864d1900163eba,m3y3x605,squad,gpt35,0
4,5733fd66d058e614000b6737,m3y3x605,squad,gpt35,1
5,572a064a3f37b3190047865f,m3y3x605,squad,gpt35,0
6,57109275b654c5140001f9a3,m3y3x605,squad,gpt35,0
7,572a12386aef051400155234,m3y3x605,squad,gpt35,0
8,57309ef18ab72b1400f9c602,m3y3x605,squad,gpt35,0
9,57287d4a2ca10214002da3e7,m3y3x605,squad,gpt35,0
10,571097baa58dae1900cd6a9b,m3y3x605,squad,gpt35,1
11,57300a9a04bcaa1900d77065,m3y3x605,squad,gpt35,1
12,572826634b864d19001645bf,m3y3x605,squad,gpt35,0
13,572fb059947a6a140053cb81,m3y3x605,squad,gpt35,0
14,5729ea263f37b319004785bd,m3y3x605,squad,gpt35,1
15,57264d58f1498d1400e8db7c,m3y3x605,squad,gpt35,0
16,5727502f708984140094dc09,m3y3x605,squad,gpt35,1
17,57097c8fed30961900e841f2,m3y3x605,squad,gpt35,0
18,57264a8cdd62a815002e808f,m3y3x605

# Load CSV and evaluate agreement

In [102]:
df = pd.read_csv('notebooks/paper_evals/23-11-24-accuracy-evaluation.csv', index_col=None)

if df.dataset.nunique() > 1:
    raise NotImplementedError('Eval only for one dataset for now')
    
def remove_incomplete(df):
    # filter out incomplete data!
    ignore = []
    for metric, mdf in df.groupby('truth_metric'):
        if (mdf.truth_value == 'FILL_IN').any():
            ignore.append(metric)
    print(f'Ignoring metrics {ignore} for now.')
    df = df[df.truth_metric.map(lambda x: x not in ignore)]
    return df

df = remove_incomplete(df)

df

Ignoring metrics ['human_jansen', 'human_sebhar'] for now.


,index,data_id,wandb_id,dataset,truth_metric,truth_value
0,0,57338007d058e614000b5bdb,m3y3x605,squad,gpt35,0
1,1,571cc5c45efbb31900334dde,m3y3x605,squad,gpt35,0
2,2,5733a5f54776f41900660f46,m3y3x605,squad,gpt35,1
3,3,5727dd2e4b864d1900163eba,m3y3x605,squad,gpt35,0
4,4,5733fd66d058e614000b6737,m3y3x605,squad,gpt35,1
...,...,...,...,...,...,...
395,95,57107d73b654c5140001f91e,uvfbxm6d,squad,llama2,0
396,96,5728661e2ca10214002da2e9,uvfbxm6d,squad,llama2,0
397,97,56e17a7ccd28a01900c679a1,uvfbxm6d,squad,llama2,1
398,98,571c97e2dd7acb1400e4c121,uvfbxm6d,squad,llama2,0


In [117]:
# compute rater agreement

pdf = df.pivot(index='index', columns='truth_metric', values='truth_value')


def agreement(x, y):
    return np.mean(x == y)

for method in ['pearson', 'kendall', 'spearman', agreement, cohen_kappa_score]:
    print(f'method: {method}')
    display(pdf.corr(method=method))

method: pearson


truth_metric,gpt35,llama2
truth_metric,,
gpt35,1.000000,0.714414
llama2,0.714414,1.000000


method: kendall


truth_metric,gpt35,llama2
truth_metric,,
gpt35,1.000000,0.714414
llama2,0.714414,1.000000


method: spearman


truth_metric,gpt35,llama2
truth_metric,,
gpt35,1.000000,0.714414
llama2,0.714414,1.000000


method: <function agreement at 0x7f6a46662a20>


truth_metric,gpt35,llama2
truth_metric,,
gpt35,1.00,0.86
llama2,0.86,1.00


method: <function cohen_kappa_score at 0x7f6a45daccc0>


truth_metric,gpt35,llama2
truth_metric,,
gpt35,1.000000,0.688474
llama2,0.688474,1.000000
